# EV Charging Network in Canada — Explore
*DATA 552 – Group 4*

This notebook performs exploratory analysis on the cleaned EV charging station dataset (`ev_chargers_geo.csv`, 1,719 stations).

**Goals:**
- Summarise the current distribution of fast chargers by province and city
- Identify which networks and facility types dominate
- Visualise station locations on a Canadian map

**Primary audience:** Technical planning team  
**Data source:** Kaggle – "EV Charging Stations in Canada – October 1, 2022"

In [13]:
import os
import pandas as pd
import numpy as np
import altair as alt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
alt.data_transformers.disable_max_rows()
os.makedirs("docs/img", exist_ok=True)


df = pd.read_csv("ev_chargers_geo.csv")
df.head()

,id,station_name,street_address,city,province,latitude,longitude,ev_network,ev_dc_fast_count,ev_level2_evse_num,ev_connector_types,groups_with_access_code,access_days_time,open_date
0,129454,Essex - Sports Complex,60 Fairview Ave W.,Essex,Ontario,42.167112,-82.816715,FLO,2,0.0,CHADEMO J1772COMBO,Public,24 hours daily,2019-07-17
1,83915,BVD Petroleum - Comber - Tesla Supercharger,7018 Industrial Drive,Comber,Ontario,42.239116,-82.550227,Tesla,8,0.0,TESLA,Public,24 hours daily; for Tesla use only,2015-04-01
2,83263,Reaume Chevrolet Buick GMC,500 Front Rd,Windsor,Ontario,42.241722,-83.102857,Non-Networked,1,1.0,J1772 J1772COMBO,Public,24 hours daily,2013-01-15
3,156672,Hydro One Inc,5001 Concession Rd 8,Old Castle,Ontario,42.243330,-82.947170,Non-Networked,2,0.0,CHADEMO J1772COMBO,PLANNED - not yet accessible (Public),Unknown,NaN
4,210228,Shell Canada Ltd,5501 Ojibway Pky,Windsor,Ontario,42.260096,-83.084966,Non-Networked,2,0.0,Unknown,PLANNED - not yet accessible (Public),Unknown,NaN


## 1. Province-Level Summary

How are fast chargers distributed across Canadian provinces?

In [14]:
prov = (
    df.groupby("province")
    .agg(
        num_stations=("id", "count"),
        total_dc_fast=("ev_dc_fast_count", "sum"),
        total_level2=("ev_level2_evse_num", "sum"),
    )
    .sort_values("total_dc_fast", ascending=False)
    .reset_index()
)
prov["total_chargers"] = prov["total_dc_fast"] + prov["total_level2"]
print(prov.to_string(index=False))

chart_stations = (
    alt.Chart(prov)
    .mark_bar(color="#4C72B0")
    .encode(
        x=alt.X("num_stations:Q", title="Number of Stations"),
        y=alt.Y("province:N", sort="-x", title=None),
        tooltip=["province", "num_stations", "total_dc_fast", "total_level2"],
    )
    .properties(title="EV Charging Stations per Province", width=300, height=200)
)

chart_fast = (
    alt.Chart(prov)
    .mark_bar(color="#DD8452")
    .encode(
        x=alt.X("total_dc_fast:Q", title="Total DC Fast Charger Ports"),
        y=alt.Y("province:N", sort="-x", title=None),
        tooltip=["province", "total_dc_fast", "num_stations"],
    )
    .properties(title="DC Fast Charger Ports per Province", width=300, height=200)
)
fig_prov = (chart_stations | chart_fast)
fig_prov.save("docs/img/fig_province_summary.png")

fig_prov

            province  num_stations  total_dc_fast  total_level2  total_chargers
             Ontario           532           1421         174.0          1595.0
              Quebec           559           1381         154.0          1535.0
    British Columbia           347            966          70.0          1036.0
             Alberta            77            170          33.0           203.0
        Saskatchewan            33            103           9.0           112.0
       New Brunswick            44             92          28.0           120.0
            Manitoba            34             87           7.0            94.0
        Newfoundland            35             54           0.0            54.0
         Nova Scotia            23             40          12.0            52.0
               Yukon            28             28           4.0            32.0
Prince Edward Island             7             14           5.0            19.0


alt.HConcatChart(...)

## 2. Top Cities by Fast Charger Count

In [15]:
city = (
    df.groupby(["city", "province"])
    .agg(
        num_stations=("id", "count"),
        total_dc_fast=("ev_dc_fast_count", "sum"),
        total_level2=("ev_level2_evse_num", "sum"),
    )
    .reset_index()
    .sort_values("total_dc_fast", ascending=False)
)
top20 = city.head(20).copy()
top20["city_label"] = top20["city"] + " (" + top20["province"].str[:2] + ")"

chart = (
    alt.Chart(top20)
    .mark_bar()
    .encode(
        x=alt.X("total_dc_fast:Q", title="Total DC Fast Charger Ports"),
        y=alt.Y("city_label:N", sort="-x", title=None),
        color=alt.Color(
            "total_dc_fast:Q",
            scale=alt.Scale(scheme="blues"),
            legend=None,
        ),
        tooltip=["city", "province", "total_dc_fast", "total_level2", "num_stations"],
    )
    .properties(
        title="Top 20 Cities by DC Fast Charger Count",
        width=400,
        height=300,
    )
)
chart.save("docs/img/fig_top20_cities.png")
display(chart)

print(f"\nTotal cities with at least one station: {len(city)}")
print(f"Top 5 cities account for {top20.head(5)['total_dc_fast'].sum() / city['total_dc_fast'].sum():.1%} of all DC fast ports")

alt.Chart(...)


Total cities with at least one station: 813
Top 5 cities account for 8.1% of all DC fast ports


## 3. Charger Count Distribution

How many DC fast charger ports does a typical station have?

In [16]:
dc_counts = df["ev_dc_fast_count"].dropna()

stats_df = pd.DataFrame({
    "stat": ["Median", "Mean"],
    "value": [dc_counts.median(), dc_counts.mean()],
    "color": ["red", "orange"],
})

histogram = (
    alt.Chart(df.dropna(subset=["ev_dc_fast_count"]))
    .mark_bar(color="#4C72B0", opacity=0.85)
    .encode(
        x=alt.X(
            "ev_dc_fast_count:Q",
            bin=alt.Bin(step=1),
            title="DC Fast Charger Ports per Station",
        ),
        y=alt.Y(
            "count():Q",
            scale=alt.Scale(type="log"),
            title="Number of Stations (log scale)",
        ),
        tooltip=[
            alt.Tooltip("ev_dc_fast_count:Q", bin=alt.Bin(step=1), title="Ports"),
            alt.Tooltip("count():Q", title="Stations"),
        ],
    )
)

fig_dist = (
    histogram.properties(
        title="Distribution of DC Fast Charger Ports per Station",
        width=650,
        height=380,
    )
)
fig_dist.save("docs/img/fig_port_distribution.png")
display(fig_dist)
print(dc_counts.describe().round(2))

print(f"\nStations with 1 port: {(dc_counts == 1).sum()} ({(dc_counts == 1).mean():.1%})")
print(f"Stations with 4 or more ports: {(dc_counts >= 4).sum()} ({(dc_counts >= 4).mean():.1%})")

alt.Chart(...)

count    1719.00
mean        2.53
std         2.89
min         1.00
25%         1.00
50%         2.00
75%         2.00
max        43.00
Name: ev_dc_fast_count, dtype: float64

Stations with 1 port: 672 (39.1%)
Stations with 4 or more ports: 280 (16.3%)


## 4. Network Operator Analysis

Which charging networks dominate the Canadian fast-charging market?

In [17]:
network = (
    df.groupby("ev_network")
    .agg(
        num_stations=("id", "count"),
        total_dc_fast=("ev_dc_fast_count", "sum"),
    )
    .sort_values("total_dc_fast", ascending=False)
    .reset_index()
)

top_n = 10
top_networks = network.head(top_n).copy()
other_row = pd.DataFrame({
    "ev_network": ["Other"],
    "num_stations": [network.iloc[top_n:]["num_stations"].sum()],
    "total_dc_fast": [network.iloc[top_n:]["total_dc_fast"].sum()],
})
network_plot = pd.concat([top_networks, other_row], ignore_index=True)
network_plot["is_other"] = network_plot["ev_network"] == "Other"

chart = (
    alt.Chart(network_plot)
    .mark_bar()
    .encode(
        x=alt.X("total_dc_fast:Q", title="Total DC Fast Charger Ports"),
        y=alt.Y("ev_network:N", sort="-x", title=None),
        color=alt.condition(
            alt.datum.is_other,
            alt.value("#AACCEE"),
            alt.value("#4C72B0"),
        ),
        tooltip=["ev_network", "total_dc_fast", "num_stations"],
    )
    .properties(
        title="DC Fast Charger Ports by Network Operator (Top 10 + Other)",
        width=400,
        height=270,
    )
)
chart.save("docs/img/fig_network_operators.png")
display(chart)

print("Top 5:")
print(network.head(5)[["ev_network", "num_stations", "total_dc_fast"]].to_string(index=False))
market_share = network.head(5)["total_dc_fast"].sum() / network["total_dc_fast"].sum()
print(f"\nTop 5 networks control {market_share:.1%} of all ports.")

alt.Chart(...)

Top 5:
        ev_network  num_stations  total_dc_fast
             Tesla           149           1415
     Non-Networked           535           1102
Circuit électrique           360            747
               FLO           234            317
               IVY            49            141

Top 5 networks control 85.4% of all ports.


## 5. Geographic Distribution — Interactive Map


In [18]:
import plotly.express as px

map_df = df[["station_name", "city", "province", "latitude", "longitude",
             "ev_dc_fast_count", "ev_network"]].dropna(subset=["latitude", "longitude"])

fig = px.scatter_mapbox(
    map_df,
    lat="latitude",
    lon="longitude",
    color="ev_dc_fast_count",
    size="ev_dc_fast_count",
    size_max=18,
    color_continuous_scale="Blues",
    zoom=3,
    center={"lat": 56, "lon": -96},
    mapbox_style="carto-positron",
    hover_data={
        "station_name": True,
        "city": True,
        "province": True,
        "ev_network": True,
        "ev_dc_fast_count": True,
        "latitude": False,
        "longitude": False,
    },
    labels={"ev_dc_fast_count": "DC Fast Ports"},
    title="EV Fast-Charging Stations in Canada",
    height=560,
)
fig.update_layout(
    coloraxis_colorbar=dict(title="DC Fast Ports"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()


## Key Findings


Ontario and British Columbia dominate hold the majority of DC fast charger ports, reflecting their larger EV markets

Most stations are small. The median station has only a small number of ports; a few large highway hubs account for a disproportionate share of capacity

Network concentration - A handful of networks control the majority of fast charger infrastructure — new entrants must compete with established players or fill geographic gaps they have not reached

Geographic clustering - Stations concentrate in the Golden Horseshoe, Metro Vancouver, and other urban cores; large stretches of the Prairies, Northern Ontario and Atlantic Canada have sparse or no coverage

These patterns directly motivate our next step: quantifying which cities are under-served given actual EV ownership levels — the focus of the scoring notebook.